In [ ]:
# !rm -rf /content/Tetris_RL
# !rm -rf /content/drive/MyDrive/tetris_dueling_sb3_logs_V1
# !rm /content/tetris_register.py
# !rm /content/Tetris_RL.zip

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:
!pip install gymnasium stable-baselines3[extra] pygame

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 18.4 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.3.0
    Uninstalling gymnasium-1.3.0:
      Successfully uninstalled gymnasium-1.3.0


In [ ]:
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# !unzip Tetris_RL.zip -d .

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
sys.path.append("/content/drive/MyDrive/Tetris_RL")
# add project to python path, allows python to find the files while importing

In [4]:
from tetris.tetris_env import TetrisGymEnv
from tetris.tetris_engine import TetrisEngine


In [ ]:
# sample=TetrisGymEnv(observation_type="board_and_heuristics")
# print(sample.survival_reward)

In [5]:
%%writefile tetris_register.py
from gymnasium.envs.registration import register

register(
    id="Tetris-v0",
    entry_point="tetris.tetris_env:TetrisGymEnv",
)

Writing tetris_register.py


In [6]:
import tetris_register

In [7]:
import os
import glob
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
import gymnasium as gym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import torch
import torch.nn as nn
from stable_baselines3.dqn.policies import DQNPolicy, QNetwork, MultiInputPolicy

class DuelingQNetwork(QNetwork):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # Override the q_net created by the parent class
        # with a dueling architecture
        net_arch = kwargs.get("net_arch", [256, 256])
        if net_arch is None or len(net_arch) == 0:
            net_arch = [256, 256]

        # Value stream
        self.value_net = nn.Sequential(
            nn.Linear(self.features_dim, net_arch[0]),
            nn.ReLU(),
            nn.Linear(net_arch[0], net_arch[1]),
            nn.ReLU(),
            nn.Linear(net_arch[1], 1)
        )

        # Advantage stream
        self.advantage_net = nn.Sequential(
            nn.Linear(self.features_dim, net_arch[0]),
            nn.ReLU(),
            nn.Linear(net_arch[0], net_arch[1]),
            nn.ReLU(),
            nn.Linear(net_arch[1], self.action_space.n)
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        features = self.extract_features(obs, self.features_extractor)
        value = self.value_net(features)
        advantages = self.advantage_net(features)
        # Combine value and advantage: Q(s,a) = V(s) + A(s,a) - mean(A(s,a))
        q_values = value + (advantages - advantages.mean(dim=1, keepdim=True))
        return q_values

class DuelingDQNPolicy(MultiInputPolicy):
    def make_q_net(self) -> DuelingQNetwork:
        net_args = self._update_features_extractor(self.net_args, features_extractor=None)
        return DuelingQNetwork(**net_args).to(self.device)

In [ ]:
# obj=TetrisGymEnv(observation_type="board_and_heuristics")
# print(obj.lines_cleared_reward, obj.step_penalty)

In [ ]:
# Create environment factory so DummyVecEnv can make it
def make_env():
    def _init():
        env = gym.make("Tetris-v0", observation_type="board_and_heuristics")
        # force the correct observation type
        env = Monitor(env)
        # print(f"step penalty = {env.step_penalty}")
        return env
    return _init


In [ ]:
# PARAMETERS - same as before
ENV_ID = "Tetris-v0"
TIMESTEPS = 300_000
EVAL_FREQ = 20_000
CHECKPOINT_FREQ = 50_000
LOG_DIR = "/content/drive/MyDrive/tetris_dueling_sb3_logs_V1"

# Find the latest checkpoint
checkpoint_dir = os.path.join(LOG_DIR, "checkpoints")
checkpoint_files = glob.glob(os.path.join(checkpoint_dir, "tetris_dueling_dqn_*.zip"))

# Recreate environments
vec_env = DummyVecEnv([make_env()])
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=False, clip_obs=10.0)

eval_env = DummyVecEnv([make_env()])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.0)
eval_env.training = False
eval_env.norm_reward = False

from stable_baselines3.common.callbacks import BaseCallback

class SaveReplayBufferCallback(BaseCallback):
    def __init__(self, save_path: str, save_freq: int, verbose=1):
        super().__init__(verbose)
        self.save_path = save_path
        self.save_freq = save_freq

    def _on_step(self) -> bool:
        if self.n_calls % self.save_freq != 0:
            return True

        path = os.path.join(self.save_path, "replay_buffer.pkl")
        self.model.save_replay_buffer(path)

        if self.verbose > 0:
            print(f"Saved replay buffer to {path}")

        return True

class SaveVecNormalizeCallback(BaseCallback):
    def __init__(self, save_path: str, save_freq: int = 0, verbose=1):
        super().__init__(verbose)
        self.save_path = save_path
        self.save_freq = save_freq # Add frequency control

    def _init_callback(self) -> None:
        # Create folder if it doesn't exist
        if self.save_path is not None:
            os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self) -> bool:
        # If save_freq is 0, we assume this callback is being called
        # by another callback (like EvalCallback), so we save every time it's called.
        # If save_freq > 0, we check the step count.

        if self.save_freq > 0:
            if self.n_calls % self.save_freq != 0:
                return True

        path = os.path.join(self.save_path, "vecnormalize.pkl")

        # SB3 environments are wrapped. We need to access the original VecNormalize
        # usually usually accessible via self.training_env
        self.training_env.save(path)

        if self.verbose > 1:
            print(f"Saved VecNormalize to {path}")
        return True

if checkpoint_files:
    # Sort by modification time to get the latest
    latest_checkpoint = max(checkpoint_files, key=os.path.getmtime)
    print(f"Found latest checkpoint: {latest_checkpoint}")

    # Extract the step number from filename
    import re
    match = re.search(r'tetris_dueling_dqn_(\d+)_steps', os.path.basename(latest_checkpoint))
    if match:
        completed_steps = int(match.group(1))
        remaining_steps = TIMESTEPS - completed_steps
        print(f"Completed steps: {completed_steps}, Remaining: {remaining_steps}")
    else:
        remaining_steps = TIMESTEPS
        print("Could not parse step count, training full duration")

    # Load the model
    model = DQN.load(
        latest_checkpoint,
        env=vec_env,
        device="auto"
    )

    replay_path = os.path.join(checkpoint_dir, "replay_buffer.pkl")
    if os.path.exists(replay_path):
        model.load_replay_buffer(replay_path)
        print("Replay buffer loaded")
    else:
        print("No replay buffer found, starting with empty buffer")

    # Load VecNormalize stats if they exist
    vecnorm_path = os.path.join(checkpoint_dir, "vecnormalize.pkl")
    if os.path.exists(vecnorm_path):
        vec_env = VecNormalize.load(vecnorm_path, vec_env)
        print(f"Loaded VecNormalize stats from {vecnorm_path}")

        # Sync eval env
        eval_env = VecNormalize.load(vecnorm_path, eval_env)
        eval_env.training = False
        eval_env.norm_reward = False
    else:
        print("No VecNormalize stats found, using fresh normalization")

    # Update model's environment
    model.set_env(vec_env)

    print("Resuming training...")

else:
    print("No checkpoints found, starting fresh training")
    remaining_steps = TIMESTEPS

    # Create new model with your custom feature extractor
    from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
    import torch as th
    import torch.nn as nn

    class TetrisCombinedExtractor(BaseFeaturesExtractor):
        def __init__(self, observation_space, features_dim: int = 512):
            super().__init__(observation_space, features_dim)

            board_shape = observation_space['board'].shape
            feature_shape = observation_space['features'].shape

            self.cnn = nn.Sequential(
                nn.Conv2d(board_shape[0], 32, kernel_size=3, stride=1, padding=1),
                nn.ReLU(),
                nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
                nn.ReLU(),
                nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
                nn.ReLU(),
                nn.Flatten()
            )

            with th.no_grad():
                dummy_board = th.zeros(1, *board_shape, dtype=th.float32)
                conv_out = self.cnn(dummy_board).shape[1]

            self.mlpf = nn.Sequential(
                nn.Linear(feature_shape[0], 64),
                nn.ReLU(),
                nn.Linear(64, 64),
                nn.ReLU(),
            )

            combined_input_size = conv_out + 64
            self.final_mlp = nn.Sequential(
                nn.Linear(combined_input_size, 512),
                nn.ReLU(),
                nn.Linear(512, features_dim),
                nn.ReLU()
            )

            self._features_dim = features_dim

        def forward(self, observations):
            board = observations['board']
            feats = observations['features']

            if not isinstance(board, th.Tensor):
                board = th.tensor(board).to(next(self.parameters()).device)
            if not isinstance(feats, th.Tensor):
                feats = th.tensor(feats).to(next(self.parameters()).device)

            conv_out = self.cnn(board)
            mlp_out = self.mlpf(feats)

            combined = th.cat([conv_out, mlp_out], dim=1)
            return self.final_mlp(combined)

    policy_kwargs = dict(
        features_extractor_class=TetrisCombinedExtractor,
        features_extractor_kwargs=dict(features_dim=512),
    )

    model = DQN(
        policy=DuelingDQNPolicy,
        env=vec_env,
        learning_rate=1e-4,
        buffer_size=1_000_000,
        learning_starts=50_000,
        batch_size=256,
        gamma=0.99,
        train_freq=(4, "step"),
        target_update_interval=5000,
        exploration_fraction=0.6,
        exploration_final_eps=0.05,
        policy_kwargs=policy_kwargs,
        verbose=1,
        tensorboard_log=LOG_DIR,
        device="auto",
    )

# Recreate callbacks
checkpoint_cb = CheckpointCallback(
    save_freq=CHECKPOINT_FREQ,
    save_path=checkpoint_dir,
    name_prefix="tetris_dueling_dqn"
)

save_replay_cb = SaveReplayBufferCallback(
    save_path=checkpoint_dir,
    save_freq=CHECKPOINT_FREQ
)

save_vec_best = SaveVecNormalizeCallback(
    save_path=os.path.join(LOG_DIR, "best_model")
)

save_vec_periodic = SaveVecNormalizeCallback(
    save_path=checkpoint_dir,
    save_freq=CHECKPOINT_FREQ
)

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(LOG_DIR, "best_model"),
    log_path=os.path.join(LOG_DIR, "eval_logs"),
    eval_freq=EVAL_FREQ,
    n_eval_episodes=8,
    deterministic=True,
    render=False,
    callback_on_new_best=save_vec_best
)

# Resume/Continue training
model.learn(
    total_timesteps=remaining_steps,
    callback=[checkpoint_cb, eval_cb, save_vec_periodic, save_replay_cb],
    reset_num_timesteps=False  # Important: don't reset the step counter
)

# Save final model
model.save(os.path.join(LOG_DIR, "tetris_dueling_dqn_final"))
vec_env.save(os.path.join(LOG_DIR, "vecnormalize.pkl"))

print("Training complete. Artifacts saved to:", LOG_DIR)

In [ ]:
import os
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# 1. Define the base path to your Drive folder
# Adjust 'Tetris_RL' if your logs are in a specific subfolder inside MyDrive
log_path = "/content/drive/MyDrive/tetris_dueling_sb3_logs_V1/"

# 2. Check if the file actually exists (Good for debugging)
vec_norm_path = os.path.join(log_path, "best_model/vecnormalize.pkl")
model_path = os.path.join(log_path, "best_model/best_model.zip") # .zip is often implicit in SB3, but good to be aware of

if not os.path.exists(vec_norm_path):
    print(f"Error: Could not find file at {vec_norm_path}")
    print("Available files in folder:", os.listdir(log_path))
else:
    # 3. Load the environment and model using the corrected paths

    # base env (assuming make_env is defined elsewhere in your code)
    dummy = DummyVecEnv([make_env()])

    # load normalization wrapper
    vec_env = VecNormalize.load(vec_norm_path, dummy)
    vec_env.training = False
    vec_env.norm_reward = False

    # load the trained model
    # Note: DQN.load handles the .zip extension automatically if omitted,
    # but the directory path must be correct.
    model = DQN.load(model_path, env=vec_env)

    print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:480: UserWarning: WARN: The environment creator metadata doesn't include `render_modes`, contains: ['render.modes']
  logger.warn(


Model loaded successfully!


In [ ]:
obs = vec_env.reset()
done = False
total_reward = 0
total_lines_cleared = 0
action_no=1

while not done:
    action, _ = model.predict(obs, deterministic=True)
    print(f"action {action_no}:", action)
    action_no+=1
    obs, reward, done, info = vec_env.step(action)
    print("reward", reward)
    total_reward += reward
    total_lines_cleared += info[0]['lines_cleared']
    vec_env.envs[0].render()

print("Episode finished. Total reward:", total_reward)
print("Lines cleared:", total_lines_cleared)



Streaming output truncated to the last 5000 lines.
##########
##########
##########
##########
##########
##..######
###..###.#
....##.#.#
....##.#..
...###.#.#
##.###....
----------


action 121: [3]
reward [0.01]
----------


##########
####..####
###..#####
##########
##########
##########
##########
##########
##########
##########
##########
##########
##########
##########
##..######
###..###.#
....##.#.#
....##.#..
...###.#.#
##.###....
----------


action 122: [3]
reward [0.01]
----------


##########
##########
####..####
###..#####
##########
##########
##########
##########
##########
##########
##########
##########
##########
##########
##..######
###..###.#
....##.#.#
....##.#..
...###.#.#
##.###....
----------


action 123: [3]
reward [0.01]
----------


##########
##########
##########
####..####
###..#####
##########
##########
##########
##########
##########
##########
##########
##########
##########
##..######
###..###.#
....##.#.#
....##.#..
...###.#.#
##.###....
